# Diffusion Models as High-Energy Physics Surrogates

**Introduction:** Long Chen  
**Code Tutorial:** Xuan Tung Nguyen  
**Affiliation:** Scientific Computing (SciComp), RPTU (Rheinland-Pfälzische Technische Universität Kaiserslautern-Landau)

**Tutorial Overview**

The objective of this tutorial is to introduce the fundamental concepts behind modern generative models, with a particular focus on Denoising Diffusion Probabilistic Models (DDPMs), and to demonstrate how these models can be applied to simulation and reconstruction problems in High-Energy Physics.

In this notebook, we build and train a simple conditional DDPM in PyTorch and apply it to calorimeter shower generation. By the end of the tutorial, participants will understand the diffusion process, train a diffusion model from scratch, resume training from checkpoints, and generate samples using both DDPM and DDIM sampling.

**Learning Objectives**

By the end of this tutorial, you will be able to:

- understand the forward and reverse diffusion processes
- train a conditional DDPM in PyTorch
- save and load checkpoints
- resume training from a checkpoint
- generate calorimeter shower images



##### Library

In [1]:
from IPython.display import Image
from IPython.display import clear_output

import h5py
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from torchvision.utils import save_image

import time
import math
import random
import os
from collections import OrderedDict
from collections import defaultdict
from copy import deepcopy


print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")



print("CUDA:", torch.cuda.is_available())
print("MPS:", torch.backends.mps.is_available())
print(device)

False
CPU
CUDA: False
MPS: True
mps


## Overview

### Methodology


![Alt Text](Images/ddpm_scheme.png)

### Diffusion Model

Denoising Diffusion Probabilistic Models (DDPMs) are generative models that learn to reverse a gradual noising process. The model starts from structured data, adds noise step by step, and learns how to reconstruct the original sample.

### Conditional Diffusion Model

For high-energy physics applications, the diffusion model can be conditioned on physical information such as particle energy, particle type, or detector geometry. In this case, the model learns the conditional distribution

$$
p(x_0 \mid y),
$$

where $x_0$ is the original data sample and $y$ is the conditioning variable.

### Forward Process

The forward process gradually adds Gaussian noise to a clean sample over $T$ timesteps. It is defined as

$$
q(x_t \mid x_{t-1}) = \mathcal{N}\left(\sqrt{1-\beta_t}\,x_{t-1}, \beta_t I\right),
$$

where $\beta_t$ is the noise schedule at timestep $t$.

A noisy sample can also be written directly as

$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon,
$$

where

$$
\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t}\alpha_s, \qquad \epsilon \sim \mathcal{N}(0, I).
$$

### Reverse Process

The reverse process learns how to remove noise step by step and recover a realistic sample from a noisy input. It is modeled as

$$
p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\left(\mu_\theta(x_t, t), \Sigma_\theta(x_t, t)\right).
$$

In practice, the neural network is trained to predict the noise added at each timestep:

$$
L_{\text{simple}} = \mathbb{E}_{x_0, t, \epsilon}\left[\left\|\epsilon - \epsilon_\theta(x_t, t, y)\right\|^2\right].
$$

### Sampling Strategy

#### DDPM Sampling

DDPM sampling starts from pure Gaussian noise,

$$
x_T \sim \mathcal{N}(0, I),
$$

and iteratively denoises the sample:

$$
x_T \rightarrow x_{T-1} \rightarrow \cdots \rightarrow x_0.
$$

At each step, the model predicts the noise $\epsilon_\theta(x_t, t, y)$ and uses it to estimate the previous state. DDPM sampling is stochastic and usually requires many denoising steps.

#### DDIM Sampling

DDIM uses the same trained model but a faster sampling procedure. Instead of following every diffusion step, it can skip steps and generate samples with fewer iterations.

First, we estimate the clean sample:

$$
\hat{x}_0 =
\frac{x_t - \sqrt{1-\bar{\alpha}_t}\,\epsilon_\theta(x_t,t,y)}
{\sqrt{\bar{\alpha}_t}}.
$$

Then the update rule is

$$
x_{t-1}
=
\sqrt{\bar{\alpha}_{t-1}}\,\hat{x}_0
+
\sqrt{1-\bar{\alpha}_{t-1}-\sigma_t^2}\,\epsilon_\theta(x_t,t,y)
+
\sigma_t z,
\qquad z \sim \mathcal{N}(0,I).
$$

When $\eta = 0$, DDIM becomes deterministic and can generate samples much faster than DDPM.

### Studying Case: Homogenous Electromagnetic Calorimeter

![Alt Text](Images/calo_for_ddpm_neww.png)

## Data Preparation

Data Preprocession

In [13]:
class HDF5Dataset(Dataset):

    def __init__(self, h5_file_path, transform=None):
        self.transform = transform

        # ============================================================
        # TODO:
        # Open the HDF5 file.
        # Load the XZ shower images.
        # Load the energy labels.
        # Convert the energy labels into integer indices.
        # ============================================================
        
    def __len__(self):
        
        # ============================================================        
        # TODO:
        # Return the number of events.
        # ============================================================
        
        pass

    def __getitem__(self, idx):
        
        # ============================================================
        # TODO:
        # Load one shower image.
        # Convert it to a float32 PyTorch tensor.
        # Add the channel dimension.
        # Apply the transform (if provided).
        # Normalize the image from [0,1] to [-1,1].
        # Load the corresponding energy label.
        # ============================================================

        return img, energy

## Diffusion Process

### Forward Diffusion

During training, we start from a clean calorimeter shower image $x_0$ and gradually add Gaussian noise.

At diffusion step $t$, the noisy image is obtained by

$$
x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon,
$$

where

- $x_0$ is the original shower image,
- $\epsilon \sim \mathcal{N}(0, I)$ is Gaussian noise,
- $\bar{\alpha}_t$ is the cumulative product of the noise schedule,
- $t$ is the diffusion timestep.

As $t$ increases, more noise is added to the image. After many diffusion steps, $x_t$ becomes almost indistinguishable from pure Gaussian noise.

### Noise Prediction

The neural network is trained to predict the Gaussian noise that was added during the forward diffusion process.

Given a noisy image $x_t$, the diffusion timestep $t$, and the conditioning energy label $E$, the model predicts

$$
\epsilon_\theta(x_t, t, E).
$$

Rather than reconstructing the clean image directly, the model learns to estimate the added noise.

The training objective is the mean squared error (MSE) between the true noise and the predicted noise:

$$
\mathcal{L}
=
\left\|
\epsilon
-
\epsilon_\theta(x_t, t, E)
\right\|^2.
$$

Once the model can accurately predict the noise, this prediction is used during the reverse diffusion process to gradually remove noise and recover a clean shower image.

In [14]:
def extract(v, t, x_shape):
    device = t.device
    out = torch.gather(v, index=t, dim=0).float().to(device)
    return out.view([t.shape[0]] + [1] * (len(x_shape) - 1))

In [15]:
class GaussianDiffusionTrainer(nn.Module):

    def __init__(self, model, beta_1, beta_T, T):
        super().__init__()

        self.model = model
        self.T = T

        # ============================================================
        # TODO:
        # Create the beta schedule.
        # Compute alpha_t = 1 - beta_t.
        # Compute the cumulative product alpha_bar_t.
        # Store:
        #       sqrt(alpha_bar_t)
        #       sqrt(1 - alpha_bar_t)
        # ============================================================

    def forward(self, x_0, energy_labels):

        # ============================================================
        # TODO:
        # Sample a random timestep t.
        # Sample Gaussian noise.
        #
        # Perform forward diffusion:
        #
        # x_t =
        # sqrt(alpha_bar_t) * x_0
        # +
        # sqrt(1-alpha_bar_t) * noise
        #
        # Predict the noise using the diffusion model.
        # Compute the MSE loss between
        # predicted noise and true noise.
        # ============================================================

        return loss

### DDPM (Gaussian) Sampling

At generation time, we start from pure Gaussian noise:

$$
x_T \sim \mathcal{N}(0, I)
$$

Then the model gradually denoises the sample one step at a time:

$$
x_T \rightarrow x_{T-1} \rightarrow \cdots \rightarrow x_0
$$

In DDPM, the reverse process is modeled as a Gaussian distribution:

$$
p_\theta(x_{t-1}\mid x_t) =
\mathcal{N}\left(\mu_\theta(x_t,t), \sigma_t^2 I\right)
$$

where the network predicts the noise $\epsilon_\theta(x_t,t)$. Using this prediction, we compute the mean of the reverse Gaussian:

$$
\mu_\theta(x_t,t)
=
\frac{1}{\sqrt{\alpha_t}}
\left(
x_t
-
\frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}
\,\epsilon_\theta(x_t,t)
\right)
$$

Then the previous sample is obtained by adding Gaussian noise:

$$
x_{t-1} = \mu_\theta(x_t,t) + \sigma_t z,
\qquad z \sim \mathcal{N}(0,I)
$$

This means DDPM sampling is stochastic: even if we start from the same noise, the generated sample can be slightly different each time because random noise is added at every denoising step.

In the code, this corresponds to:

- predicting noise with the network,
- computing the reverse-process mean,
- adding sampled noise when $t > 0$.

DDPM sampling is the standard diffusion sampling procedure, but it is usually slower because it uses many timesteps.

In [16]:
class GaussianDiffusionSampler(nn.Module):

    def __init__(self, model, beta_1, beta_T, T, w=0.0):
        super().__init__()

        self.model = model
        self.T = T
        self.w = w

        # ============================================================
        # TODO:
        # Compute the diffusion schedule.
        #
        # Store:
        #   - beta_t
        #   - alpha_t
        #   - alpha_bar_t
        #   - coefficients for the reverse process
        #   - posterior variance
        # ============================================================

    def predict_xt_prev_mean_from_eps(self, x_t, t, eps):

        # ============================================================
        # TODO:
        # Compute the mean of p(x_{t-1} | x_t)
        #
        # μθ(x_t,t)
        # ============================================================

        return xt_prev_mean

    def p_mean_variance(self, x_t, t, labels):

        # ============================================================
        # TODO:
        # Compute the variance.
        # Predict εθ(x_t,t,E).
        # Compute the reverse-process mean.
        # ============================================================

        return mean, var

    def forward(self, x_T, labels):

        # ============================================================
        # Start from Gaussian noise.
        # ============================================================

        x_t = x_T

        # ============================================================
        # Reverse diffusion
        #
        # x_T -> x_{T-1} -> ... -> x_0
        # ============================================================

        for time_step in reversed(range(self.T)):

            # TODO:
            # Build the timestep tensor.
            # Compute mean and variance.
            # Sample Gaussian noise
            # Compute x_{t-1}

        return torch.clip(x_t, -1, 1)

### DDIM Sampling

DDIM uses the same trained diffusion model, but with a more efficient sampling procedure.

Like DDPM, we start from pure Gaussian noise:

$$
x_T \sim \mathcal{N}(0, I)
$$

However, DDIM does not need to follow every single diffusion step. Instead, it can use a smaller set of timesteps and generate samples more quickly.

First, the model predicts the noise $\epsilon_\theta(x_t,t)$. From that, we estimate the clean image:

$$
\hat{x}_0
=
\frac{x_t - \sqrt{1-\bar{\alpha}_t}\,\epsilon_\theta(x_t,t)}
{\sqrt{\bar{\alpha}_t}}
$$

Then we move to a previous timestep using this estimate.

A common DDIM update can be written as:

$$
x_{t-1}
=
\sqrt{\bar{\alpha}_{t-1}}\,\hat{x}_0
+
\sqrt{1-\bar{\alpha}_{t-1}-\sigma_t^2}\,\epsilon_\theta(x_t,t)
+
\sigma_t z,
\qquad z \sim \mathcal{N}(0,I)
$$

where $\sigma_t$ is controlled by the parameter $\eta$.

If $\eta = 0$, then $\sigma_t = 0$, and DDIM becomes deterministic:

$$
x_{t-1}
=
\sqrt{\bar{\alpha}_{t-1}}\,\hat{x}_0
+
\sqrt{1-\bar{\alpha}_{t-1}}\,\epsilon_\theta(x_t,t)
$$

This makes DDIM much faster than DDPM, because it can generate samples with far fewer denoising steps.

In summary:

- DDPM: stochastic sampling, usually slower, uses every diffusion step.
- DDIM: faster sampling, can skip steps, and can be deterministic when $\eta = 0$.

Both methods use the same trained network; only the sampling procedure is different.

In [17]:
class DDIMSampler(nn.Module):

    def __init__(self, model, beta_1, beta_T, T,
                 eta=0.0, ddim_steps=50):
        super().__init__()

        self.model = model
        self.T = T
        self.eta = eta
        self.ddim_steps = ddim_steps

        # ============================================================
        # TODO:
        # Build the diffusion schedule.
        #
        # Store:
        #   - beta_t
        #   - alpha_t
        #   - alpha_bar_t
        #   - sqrt(alpha_bar_t)
        #   - sqrt(1 - alpha_bar_t)
        # ============================================================

        # ============================================================
        # TODO:
        # Choose the timesteps used for DDIM sampling.
        # ============================================================

    def get_ddim_timesteps(self):

        # ============================================================
        # TODO:
        # Return evenly spaced timesteps.
        # ============================================================

        pass

    def forward(self, x, energy_labels):

        # ============================================================
        # Reverse diffusion using DDIM.
        # ============================================================

        for i in reversed(range(len(self.ddim_timesteps))):

            # ============================================================
            # TODO
            # Predict the noise:
            # Estimate the clean image:
            # Compute the previous timestep.
            # Compute σ (controlled by η).
            # DDIM update:
            # ============================================================
            
        return x

## Model Design

![Alt Text](Images/unet.jpg)

In [18]:
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

We don't learn the timestep directly. Instead, we first create a sinusoidal positional encoding, just like in the Transformer, and then refine it with a small MLP.

In [19]:
class TimeEmbedding(nn.Module):

    def __init__(self, T, d_model, dim):
        super().__init__()

        # ============================================================
        # TODO:
        # Build the sinusoidal timestep embeddings.
        #
        # Hint:
        # 1. Compute the sinusoidal encoding.
        # 2. Create an embedding layer.
        # 3. Pass it through a small MLP.
        # ============================================================

    def forward(self, t):

        # ============================================================
        # TODO:
        # Return the timestep embedding.
        # ============================================================

        pass

In [20]:
class ConditionalEmbedding(nn.Module):

    def __init__(self, num_labels, d_model, dim):
        super().__init__()

        # ============================================================
        # TODO:
        # Create an embedding for the energy labels.
        #
        # The embedding is then processed by
        # a small MLP.
        # ============================================================

    def forward(self, labels):

        # ============================================================
        # TODO:
        # Return the conditional embedding.
        # ============================================================

        pass

In [21]:
class DownSample(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.c1 = nn.Conv2d(in_ch, in_ch, 3, stride=2, padding=1)
        self.c2 = nn.Conv2d(in_ch, in_ch, 5, stride=2, padding=2)

    def forward(self, x, temb, cemb):
        return self.c1(x) + self.c2(x)

In [22]:
class UpSample(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.c = nn.Conv2d(in_ch, in_ch, 3, stride=1, padding=1)
        self.t = nn.ConvTranspose2d(in_ch, in_ch, 5, 2, 2, 1)

    def forward(self, x, temb, cemb):
        x = self.t(x)
        return self.c(x)

In [23]:
class AttnBlock(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.group_norm = nn.GroupNorm(32, in_ch)
        self.proj_q = nn.Conv2d(in_ch, in_ch, 1)
        self.proj_k = nn.Conv2d(in_ch, in_ch, 1)
        self.proj_v = nn.Conv2d(in_ch, in_ch, 1)
        self.proj = nn.Conv2d(in_ch, in_ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.group_norm(x)

        q = self.proj_q(h).permute(0, 2, 3, 1).view(B, H * W, C)
        k = self.proj_k(h).view(B, C, H * W)
        w = torch.bmm(q, k) * (C ** (-0.5))
        w = F.softmax(w, dim=-1)

        v = self.proj_v(h).permute(0, 2, 3, 1).view(B, H * W, C)
        h = torch.bmm(w, v).view(B, H, W, C).permute(0, 3, 1, 2)

        return x + self.proj(h)

In [24]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, dropout, attn=True):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.GroupNorm(32, in_ch),
            Swish(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        )

        self.temb_proj = nn.Sequential(
            Swish(),
            nn.Linear(tdim, out_ch),
        )

        self.cond_proj = nn.Sequential(
            Swish(),
            nn.Linear(tdim, out_ch),
        )

        self.block2 = nn.Sequential(
            nn.GroupNorm(32, out_ch),
            Swish(),
            nn.Dropout(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )

        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.attn = AttnBlock(out_ch) if attn else nn.Identity()

    def forward(self, x, temb, labels):
        h = self.block1(x)
        h = h + self.temb_proj(temb)[:, :, None, None] + self.cond_proj(labels)[:, :, None, None]
        h = self.block2(h)
        h = h + self.shortcut(x)
        return self.attn(h)

In [25]:
class UNet(nn.Module):
    def __init__(self, T, num_energy_labels, ch, ch_mult, num_res_blocks, dropout):
        super().__init__()

        tdim = ch * 4

        self.time_embedding = TimeEmbedding(T, ch, tdim)
        self.energy_embedding = ConditionalEmbedding(num_energy_labels, ch, tdim)

        self.head = nn.Conv2d(1, ch, 3, padding=1)

        self.downblocks = nn.ModuleList()
        chs = [ch]
        now_ch = ch

        for i, mult in enumerate(ch_mult):
            out_ch = ch * mult
            for _ in range(num_res_blocks):
                self.downblocks.append(ResBlock(now_ch, out_ch, tdim, dropout))
                now_ch = out_ch
                chs.append(now_ch)
            if i != len(ch_mult) - 1:
                self.downblocks.append(DownSample(now_ch))
                chs.append(now_ch)

        self.middleblocks = nn.ModuleList([
            ResBlock(now_ch, now_ch, tdim, dropout, attn=True),
            ResBlock(now_ch, now_ch, tdim, dropout, attn=False),
        ])

        self.upblocks = nn.ModuleList()
        for i, mult in reversed(list(enumerate(ch_mult))):
            out_ch = ch * mult
            for _ in range(num_res_blocks + 1):
                self.upblocks.append(ResBlock(chs.pop() + now_ch, out_ch, tdim, dropout, attn=False))
                now_ch = out_ch
            if i != 0:
                self.upblocks.append(UpSample(now_ch))

        assert len(chs) == 0

        self.tail = nn.Sequential(
            nn.GroupNorm(32, now_ch),
            Swish(),
            nn.Conv2d(now_ch, 1, 3, padding=1)
        )

    def forward(self, x, t, energy_labels):
        temb = self.time_embedding(t)
        e_emb = self.energy_embedding(energy_labels)

        h = self.head(x)
        hs = [h]

        for layer in self.downblocks:
            h = layer(h, temb, e_emb)
            hs.append(h)

        for layer in self.middleblocks:
            h = layer(h, temb, e_emb)

        for layer in self.upblocks:
            if isinstance(layer, ResBlock):
                h = torch.cat([h, hs.pop()], dim=1)
            h = layer(h, temb, e_emb)

        h = self.tail(h)
        assert len(hs) == 0
        return h

## Model Training

In [26]:
# ============================================================
# Diffusion Hyperparameters
# ============================================================

T = 500

BETA_1 = 1e-4
BETA_T = 0.028

# ============================================================
# Dataset
# ============================================================

NUM_ENERGY_LABELS = 11
IMAGE_SIZE = 32
BATCH_SIZE = 128

# ============================================================
# U-Net Architecture
# ============================================================

CH = 32
CH_MULT = [1, 2, 2, 2]
NUM_RES_BLOCKS = 2
DROPOUT = 0.1

# ============================================================
# Training
# ============================================================

LR = 1e-3
EPOCHS = 5

# ============================================================
# Checkpoints
# ============================================================

SAVE_DIR = "./checkpoint"
os.makedirs(SAVE_DIR, exist_ok=True)

In [27]:
model = UNet(
    T=T,
    num_energy_labels=NUM_ENERGY_LABELS,
    ch=CH,
    ch_mult=CH_MULT,
    num_res_blocks=NUM_RES_BLOCKS,
    dropout=DROPOUT
).to(device)

trainer = GaussianDiffusionTrainer(
    model,
    beta_1=BETA_1,
    beta_T=BETA_T,
    T=T
).to(device)

sampler = DDIMSampler(
    model,
    beta_1=BETA_1,
    beta_T=BETA_T,
    T=T,
    eta=0.0,
    ddim_steps=50
).to(device)

print("Model, trainer, and sampler are ready.")

Model, trainer, and sampler are ready.


In [28]:
dataset = HDF5Dataset(
    H5_PATH,
    transform=transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        antialias=True
    )
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

x_batch, e_batch = next(iter(loader))
print("Batch image shape:", x_batch.shape)
print("Batch energy shape:", e_batch.shape)

Batch image shape: torch.Size([128, 1, 32, 32])
Batch energy shape: torch.Size([128])


In [ ]:
# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# Track loss
loss_history = []

# Training
model.train()

for epoch in range(EPOCHS):
    t0 = time.time()
    running_loss = 0.0

    print(f"\nStarting epoch {epoch+1}/{EPOCHS}...", flush=True)

    for step, (x, energy) in enumerate(loader):
        x = x.to(device)
        energy = energy.to(device)

        loss = trainer(x, energy).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(loader)
    loss_history.append(avg_loss)

    dt = time.time() - t0
    clear_output(wait=True)

    plt.figure(figsize=(7,4))
    plt.plot(loss_history, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Average Loss")
    plt.title("Training Loss")
    plt.grid(alpha=0.3)
    plt.show()

    print(
        f"Finished epoch {epoch+1}/{EPOCHS} | "
        f"avg loss = {avg_loss:.6f} | "
        f"time = {dt:.1f}s"
    )

    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch + 1,
        "loss_history": loss_history,
    }
    torch.save(checkpoint, os.path.join(SAVE_DIR, "latest.pt"))

Get the pre-trained checkpoint

## Sampling the results

## Performance Evaluation

## References

- X.T. Nguyen et al., “Differentiable Surrogate for Detector Simulation and Design with Diffusion Models,” MLST 7, 025061 (2026). arXiv:2601.07859.

- J. Ho et al., “Denoising Diffusion Probabilistic Models,” NeurIPS 33, 6840-6851 (2020). arXiv:2006.11239.

- J. Song et al., “Denoising Diffusion Implicit Models,” ICLR 2021, (2021). arXiv:2010.02502.